In [0]:
-- ============================================
-- GOLD 层: 数据聚合
-- ============================================

-- 1. 刷新每日销售汇总
insert overwrite aws3.gold.daily_sales_summary
select
   dd.full_date,
   dd.day,
   dd.year,
   dd.quarter,
   dd.month,
   dd.is_weekend,
   dd.day_of_week,
   dd.week_of_year,
   dp.name province_name,
   dp.region_name,
   ds.sku_name,
   ds.spu_name,
   ds.category3_name,
   ds.category2_name,
   ds.category1_name,
   ds.tm_name,
   nvl(dus.name,'未知') user_name,--应该在维表时处理，以下都是
   nvl(dus.birthday,CAST('9999-12-31' AS DATE)) birthday,
   nvl(dus.gender,'未知') gender,
   nvl(dus.phone_num,'未知') phone_num,
   nvl(dus.email,'未知') email,
   nvl(dus.user_level,'未知') user_level,
   sum(f.sku_num) sku_num,
   sum(f.split_activity_amount) split_activity_amount,
   sum(f.split_coupon_amount) split_coupon_amount,
   sum(f.split_original_amount) split_original_amount,
   sum(f.split_total_amount) split_total_amount
from
    aws3.silver.fact_orders f
    left join aws3.silver.dim_date dd on f.date_id = dd.date_key
    left join aws3.silver.dim_province dp on f.province_id = dp.id
    left join aws3.silver.dim_sku ds on f.sku_id = ds.id
    left join aws3.silver.dim_user dus on f.user_id = dus.id 
        and dus.is_current = true  --当前状态的用户状态,两个只用一个
        --and f.create_time between dus.valid_from and dus.valid_to  --订单时的用户状态且是上线后的，因为没有之前状态,两个只用一个
group by 
   dd.full_date,
   dd.day,
   dd.year,
   dd.quarter,
   dd.month,
   dd.is_weekend,
   dd.day_of_week,
   dd.week_of_year,
   dp.name,
   dp.region_name,
   ds.sku_name,
   ds.spu_name,
   ds.category3_name,
   ds.category2_name,
   ds.category1_name,
   ds.tm_name,
   nvl(dus.name,'未知'),--应该在维表时处理，以下都是
   nvl(dus.birthday,CAST('9999-12-31' AS DATE)),
   nvl(dus.gender,'未知'),
   nvl(dus.phone_num,'未知'),
   nvl(dus.email,'未知'),
   nvl(dus.user_level,'未知')
;
